In [11]:
# ============================================================
# SIMPLE FINE-TUNING EXAMPLE THAT AVOIDS DATASETS/TORCHVISION
# Task: Sentiment classification
# Model: distilbert-base-uncased
# ============================================================

!pip install -q transformers torch

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW

# ------------------------------------------------------------
# 1. SMALL DATASET
# label 1 = Positive
# label 0 = Negative
# ------------------------------------------------------------

texts = [
    "This movie is excellent",
    "I loved this film",
    "The acting was fantastic",
    "This was a wonderful movie",
    "The story was very good",
    "I really enjoyed the movie",
    "Amazing film and great acting",
    "This movie made me happy",

    "This movie is terrible",
    "I hated this film",
    "The acting was very bad",
    "This was a boring movie",
    "The story was disappointing",
    "I did not enjoy the movie",
    "Awful film and terrible acting",
    "This movie was a waste of time"
]

labels = [
    1,1,1,1,1,1,1,1,
    0,0,0,0,0,0,0,0
]

In [12]:
# ------------------------------------------------------------
# 2. LOAD TOKENIZER AND PRETRAINED MODEL
# ------------------------------------------------------------

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
# ------------------------------------------------------------
# 3. CREATE PYTORCH DATASET
# ------------------------------------------------------------

class SentimentDataset(Dataset):

    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):

        encoding = self.tokenizer(
            self.texts[index],
            truncation=True,
            padding="max_length",
            max_length=64,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[index])
        }


dataset = SentimentDataset(
    texts,
    labels,
    tokenizer
)

dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True
)

In [14]:
# ------------------------------------------------------------
# 4. USE GPU IF AVAILABLE
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

model.to(device)

# ------------------------------------------------------------
# 5. OPTIMIZER
# ------------------------------------------------------------

optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

Using device: cpu


In [15]:
# ------------------------------------------------------------
# 6. FINE-TUNING
# ------------------------------------------------------------

epochs = 3

print("\nStarting fine-tuning...\n")

model.train()

for epoch in range(epochs):

    total_loss = 0

    for batch in dataloader:

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        labels_batch = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels_batch
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(dataloader)

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"- Loss: {average_loss:.4f}"
    )

print("\nFine-tuning completed!")

# ------------------------------------------------------------
# 7. TEST FUNCTION
# ------------------------------------------------------------

def predict(text):

    model.eval()

    encoding = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    input_ids = encoding["input_ids"].to(device)

    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    prediction = torch.argmax(
        outputs.logits,
        dim=1
    ).item()

    if prediction == 1:
        return "POSITIVE"

    return "NEGATIVE"

# ------------------------------------------------------------
# 8. TEST THE FINE-TUNED MODEL
# ------------------------------------------------------------

print("\nTesting model:\n")

print(
    "This movie was fantastic ->",
    predict("This movie was fantastic")
)

print(
    "This film was horrible ->",
    predict("This film was horrible")
)

print(
    "I really enjoyed the acting ->",
    predict("I really enjoyed the acting")
)


Starting fine-tuning...

Epoch 1/3 - Loss: 0.6904
Epoch 2/3 - Loss: 0.6744
Epoch 3/3 - Loss: 0.6588

Fine-tuning completed!

Testing model:

This movie was fantastic -> POSITIVE
This film was horrible -> NEGATIVE
I really enjoyed the acting -> POSITIVE


In [9]:
# ------------------------------------------------------------
# 9. SAVE MODEL
# ------------------------------------------------------------

model.save_pretrained(
    "./my_finetuned_model"
)

tokenizer.save_pretrained(
    "./my_finetuned_model"
)

print("\nFine-tuned model saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Fine-tuned model saved.
